# Gene-level coding burden: two patient halves × three nuclei sets, matched to control nuclei totals

1. The 52 patients are split into two halves of 26 (`SPLIT = "random"`; or `"paired"` = patients ranked by
   nuclei count, and each consecutive pair randomly divided between the halves, which balances nuclei counts).
2. Within each half, three sets of patient nuclei are drawn, each with **exactly as many nuclei as the
   control cohort** (5,062). Nuclei are allocated to patients in proportion to their size (largest-remainder
   rounding), with **every patient contributing ≥ 1 nucleus**. The three sets within a half are
   non-overlapping whenever the patient has enough nuclei (3 × allocation ≤ available); otherwise that
   patient's nuclei are drawn independently per set (reported).
3. Control nuclei are kept fixed (all nuclei, every set).
4. Per-sample burden (`n_count`, `total_cells`, `burden`, `n_variants`, `median_DP`, covariates) is computed
   exactly as in the original notebook.
   

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = "/home/AD/vbidhan/study232-missionbio_TDP-C/manuscript_data"
PARQUET_DIR     = f"{ROOT}/parquets"
ANNOTATIONS     = "/results/rr/study/hg38s/study232-missionbio_TDP-C/manuscript_data/Annotations_vafhet30.csv"
SAMPLE_METADATA = "/results/rr/study/hg38s/study232-missionbio_TDP-C/data/metadata_hemisphere_PMI.csv"
OUT_DIR         = f"{ROOT}/Burden_files/gene_halves"
SPLIT    = "paired"     # "random" or "paired" (random within nuclei-ranked pairs -> balanced halves)
N_SETS   = 3            # sets per half
SEED     = 1000
GENES    = ["TARDBP", "TET2", "TMEM106B", "OPTN", "TBK1", "GRN", "UNC13A"]

NON_CODING = [
    "intron_variant", "splice_polypyrimidine_tract_variant&intron_variant",
    "splice_region_variant&splice_polypyrimidine_tract_variant&intron_variant",
    "splice_donor_region_variant&intron_variant", "splice_region_variant&5_prime_UTR_variant",
    "splice_donor_5th_base_variant&intron_variant", "splice_region_variant&intron_variant",
    "3_prime_UTR_variant", "5_prime_UTR_variant",
]

## Load data

In [2]:
pq = Path(PARQUET_DIR)
NGT = pd.read_parquet(pq / "NGT_filtered_STG_final.parquet")
DP = pd.read_parquet(pq / "DP_filtered_STG_final.parquet").loc[NGT.index]
cell_meta = pd.read_parquet(pq / "metadata_filtered_repeats_germline_freq.parquet").loc[NGT.index]

ann = pd.read_csv(ANNOTATIONS, sep="\t", low_memory=False)
ann = ann[ann["variant_id"].isin(NGT.columns)]

smeta = pd.read_csv(SAMPLE_METADATA, sep=";").set_index("Sample")
smeta = smeta[smeta["tissue"] == "STG"].copy()
for c in ["AGEATDEATH", "AGEATONSET"]:
    smeta[c] = pd.to_numeric(smeta[c].astype(str).str.replace(",", ".", regex=False), errors="coerce")
COV = {"Condition": "Group", "Sex": "SEXatBirth", "Age_at_Death": "AGEATDEATH",
       "Recruitment_Site": "Recruitment_Site", "Pool": "pool", "Hemisphere": "Hemisphere",
       "AGEATONSET": "AGEATONSET"}

sid, stype = cell_meta["sample_id"], cell_meta["sample_type"]
n_avail = sid.value_counts()
group = stype.groupby(sid).first()
pat_ids = group.index[group == "P"].to_numpy()
n_ctrl_total = int((stype == "C").sum())
print(f"Nuclei: {len(NGT)} | patients: {len(pat_ids)} ({(stype == 'P').sum()} nuclei)"
      f" | controls: {(group == 'C').sum()} ({n_ctrl_total} nuclei)")

Nuclei: 39800 | patients: 52 (34738 nuclei) | controls: 26 (5062 nuclei)


## Split patients into two halves

In [3]:
rng = np.random.default_rng(SEED)
if SPLIT == "random":
    perm = rng.permutation(pat_ids)
    half = pd.Series("A", index=perm); half.iloc[len(perm) // 2:] = "B"
elif SPLIT == "paired":
    ranked = n_avail[pat_ids].sort_values(kind="stable").index.to_numpy()
    half = pd.Series("A", index=ranked)
    for i in range(0, len(ranked) - 1, 2):              # each pair: one patient to A, one to B, at random
        half.iloc[i + rng.integers(2)] = "B"
else:
    raise ValueError("SPLIT must be 'random' or 'paired'")
half = half.sort_index()

out = Path(OUT_DIR); out.mkdir(parents=True, exist_ok=True)

def balance(ids):
    m = smeta.reindex(ids)
    n = n_avail[ids]
    return pd.Series({"n_patients": len(ids), "nuclei_total": n.sum(), "nuclei_median": n.median(),
                      "age_at_death_mean": m["AGEATDEATH"].mean(), "female_n": (m["SEXatBirth"] == "F").sum(),
                      "n_pools": m["pool"].nunique(), "n_sites": m["Recruitment_Site"].nunique()})

bal = pd.DataFrame({h: balance(half.index[half == h]) for h in ["A", "B"]})
bal["Controls"] = balance(group.index[group == "C"])
bal.to_csv(out / "half_balance.tsv", sep="\t")
print(bal.round(1))
print("\nPool composition (patients per pool):")
print(pd.crosstab(smeta.reindex(half.index)["pool"], half))

                         A        B  Controls
n_patients            26.0     26.0      26.0
nuclei_total       17614.0  17124.0    5062.0
nuclei_median        352.5    326.0      91.5
age_at_death_mean     72.1     68.8      76.7
female_n              11.0      8.0      14.0
n_pools               17.0     18.0      16.0
n_sites                7.0      7.0       4.0

Pool composition (patients per pool):
col_0  A  B
pool       
4      2  1
5      2  1
6      2  1
7      2  1
8      1  2
9      2  1
10     1  2
11     2  1
12     2  1
13     1  2
14     1  1
15     1  1
16     2  0
17     0  4
18     0  1
20     1  1
22     0  1
23     2  0
24     1  1
25     1  3


## Allocate nuclei per patient and draw the sets
Allocation: proportional to each patient's nuclei, ≥ 1 per patient, summing exactly to the control total.

In [5]:
def allocate(n: np.ndarray, target: int) -> np.ndarray:
    raw = n * target / n.sum()
    a = np.minimum(np.maximum(np.floor(raw).astype(int), 1), n)
    rem = raw - np.floor(raw)
    diff = target - a.sum()
    order = np.argsort(-rem) if diff > 0 else np.argsort(rem)   # largest remainders up / smallest down
    i = 0
    while diff != 0:
        j = order[i % len(n)]
        if diff > 0 and a[j] < n[j]:
            a[j] += 1; diff -= 1
        elif diff < 0 and a[j] > 1:
            a[j] -= 1; diff += 1
        i += 1
    return a

set_id = {}                                     # set number -> boolean mask over NGT rows (patient nuclei)
alloc_tab, set_map = [], []
for h_i, h in enumerate(["A", "B"]):
    ids = half.index[half == h].to_numpy()
    n = n_avail[ids].to_numpy()
    a = allocate(n, n_ctrl_total)
    masks = [np.zeros(len(sid), bool) for _ in range(N_SETS)]
    for p, n_p, a_p in zip(ids, n, a):
        rows = np.flatnonzero(sid.to_numpy() == p)
        disjoint = N_SETS * a_p <= n_p
        perm = rng.permutation(rows)
        for s in range(N_SETS):
            pick = perm[s * a_p:(s + 1) * a_p] if disjoint else rng.choice(rows, a_p, replace=False)
            masks[s][pick] = True
        alloc_tab.append((p, h, n_p, a_p, round(a_p / n_p, 3), disjoint))
    for s in range(N_SETS):
        k = h_i * N_SETS + s + 1
        set_id[k] = masks[s]
        set_map.append((f"set_{k}", h, s + 1, int(masks[s].sum())))

alloc = pd.DataFrame(alloc_tab, columns=["sample_id", "half", "n_available", "n_per_set", "fraction", "disjoint_sets"])
alloc.to_csv(out / "patient_halves.tsv", sep="\t", index=False)
set_map = pd.DataFrame(set_map, columns=["set", "half", "set_in_half", "patient_nuclei"])
set_map.to_csv(out / "set_map.tsv", sep="\t", index=False)
print(set_map)
print("\nNuclei per patient per set (median): " +
      " | ".join(f"half {h}: {alloc.loc[alloc.half == h, 'n_per_set'].median():.0f}" for h in "AB") +
      f" | controls: {n_avail[group.index[group == 'C']].median():.1f}")
nd = (~alloc["disjoint_sets"]).sum()
print(f"Patients whose sets overlap (too few nuclei for 3 disjoint sets): {nd}")

     set half  set_in_half  patient_nuclei
0  set_1    A            1            5062
1  set_2    A            2            5062
2  set_3    A            3            5062
3  set_4    B            1            5062
4  set_5    B            2            5062
5  set_6    B            3            5062

Nuclei per patient per set (median): half A: 102 | half B: 96 | controls: 91.5
Patients whose sets overlap (too few nuclei for 3 disjoint sets): 0


In [ ]:
# Median across all sets
alloc.n_per_set.median()  

101.5

## Per-sample burden and output

In [5]:
def burden_table(cells, coding, all_vars):
    """Per-sample summary for a subset of nuclei (boolean mask); mirrors notebook cell 4."""
    g = sid[cells]
    ngt = NGT.loc[cells]
    res = pd.DataFrame({"n_count": ngt[coding].isin([1, 2]).any(axis=1).groupby(g).sum(),
                        "total_cells": g.groupby(g).size()})
    res.insert(0, "burden", (res["n_count"] / res["total_cells"]).round(6))
    res.insert(1, "Group", stype[cells].groupby(g).first())
    res.insert(3, "n_variants", ngt[all_vars].isin([1, 2]).groupby(g).any().sum(axis=1))   # all variants, as originally
    dp = DP.loc[cells, coding]
    dp.columns = dp.columns.str.split(":").str[1]
    dp = dp.loc[:, ~dp.columns.duplicated()]
    res["median_DP"] = dp.groupby(g).median().apply(lambda r: np.median(r.to_numpy()), axis=1)
    for new, old in COV.items():
        res[new] = res.index.map(smeta[old])
    res["Age_at_Death"] = pd.to_numeric(res["Age_at_Death"], errors="coerce")
    res.index.name = None
    return res


ctrl_mask = (stype == "C").to_numpy()
for gene in GENES:
    ag = ann[ann["SYMBOL"] == gene]
    all_vars = ag["variant_id"].tolist()
    coding = ag.loc[~ag["Consequence"].isin(NON_CODING), "variant_id"].tolist()
    ctrl = burden_table(ctrl_mask, coding, all_vars)
    for k, m in set_id.items():
        d = out / f"set_{k}"; d.mkdir(exist_ok=True)
        pd.concat([burden_table(m, coding, all_vars), ctrl]).to_csv(d / f"{gene}_Coding_variants.tsv", sep="\t")
    print(f"{gene}: {len(coding)} coding variants written for {len(set_id)} sets")

TARDBP: 1503 coding variants written for 6 sets
TET2: 4965 coding variants written for 6 sets
TMEM106B: 1042 coding variants written for 6 sets
OPTN: 2027 coding variants written for 6 sets
TBK1: 2491 coding variants written for 6 sets
GRN: 1449 coding variants written for 6 sets
UNC13A: 5625 coding variants written for 6 sets


## Sanity checks

In [6]:
for k, m in set_id.items():
    h = set_map.loc[k - 1, "half"]
    ids_in = set(sid[m].unique())
    assert m.sum() == n_ctrl_total, f"set_{k}: patient nuclei != control nuclei"
    assert ids_in == set(half.index[half == h]), f"set_{k}: not every patient in half {h} is represented"
    assert set(sid[m].unique()).isdisjoint(half.index[half != h]), f"set_{k}: contains patients from the other half"
for h_i in range(2):
    ks = [h_i * N_SETS + s + 1 for s in range(N_SETS)]
    overlap = np.sum(np.sum([set_id[k] for k in ks], axis=0) > 1)
    print(f"half {'AB'[h_i]}: nuclei in more than one set = {overlap}")
print("All checks passed: each set has", n_ctrl_total, "patient nuclei, all patients of its half, none from the other half.")

half A: nuclei in more than one set = 0
half B: nuclei in more than one set = 0
All checks passed: each set has 5062 patient nuclei, all patients of its half, none from the other half.
